# Data Preprocessing

Flatten collected YouTube comments and replies into rows for sentiment analysis and network analysis.


In [82]:
import json
from pathlib import Path
import pandas as pd
import random 
from langdetect import detect, LangDetectException
from collections import Counter
import nltk
import string
import re
import html
import emoji
from nltk.corpus import stopwords


PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
VIDEO_DATA_PATH = DATA_DIR / "video_data.json"
MET_GALA_ENTITIES_PATH = DATA_DIR / "met_gala_entities.json"

RANDOM_SEED = 42


In [83]:
with open(VIDEO_DATA_PATH, "r", encoding="utf-8") as f:
    video_data = json.load(f)["videos"]

print("Video data loaded")
print("Total videos:", len(video_data))
print("Total collected comment rows:", sum(len(video.get("comments", [])) for video in video_data))


Video data loaded
Total videos: 110
Total collected comment rows: 63250


In [84]:
comments_flattened = []
for video in video_data:
    video_context = {
        "video_id": video.get("videoId"),
        "video_title": video.get("title"),
        "channel_id": video.get("channelId"),
        "channel_title": video.get("channelTitle"),
        "video_published_at": video.get("publishedAt"),
        "video_view_count": video.get("viewCount", 0),
        "video_like_count": video.get("likeCount", 0),
        "video_available_comment_count": video.get("commentCount", 0),
    }

    for comment in video.get("comments", []):
        comments_flattened.append({
            **video_context,
            "comment_id": comment.get("commentId"),
            "comment_text": comment.get("text", ""),
            "comment_author_id": comment.get("authorId"),
            "comment_author": comment.get("author"),
            "comment_published_at": comment.get("publishedAt"),
            "comment_updated_at": comment.get("updatedAt"),
            "comment_like_count": comment.get("likeCount", 0),
            "is_reply": comment.get("isReply", False),
            "parent_comment_id": comment.get("parentCommentId"),
            "reply_to_author_id": comment.get("replyToAuthorId"),
            "top_level_reply_count": comment.get("totalReplyCount", 0),
            "text_length": len(comment.get("text", "") or ""),
        })

total_comments = len(comments_flattened)
total_replies = sum(1 for c in comments_flattened if c.get("is_reply") == True)
total_parent_comments = total_comments - total_replies

print(f"Flattened comment rows: {total_comments}\n")

print(f"Total comments: {total_comments}")
print(f"Total parent comments: {total_parent_comments}")
print(f"Total replies: {total_replies}")


Flattened comment rows: 63250

Total comments: 63250
Total parent comments: 48960
Total replies: 14290


In [85]:

TOTAL_RANDOM_SAMPLES = 25

print("\nRANDOM SAMPLE OF COMMENTS TO IDENTIFY ISSUES")
print("=" * 80)
random.seed(RANDOM_SEED)
random_sample_indices = random.sample(range(len(comments_flattened)), min(TOTAL_RANDOM_SAMPLES, len(comments_flattened)))
for i, idx in enumerate(random_sample_indices):
    text = comments_flattened[idx]['comment_text'].strip().replace('\n', ' ').replace('\r', '')
    text = ' '.join(text.split())
    print(f"[{i+1}/{TOTAL_RANDOM_SAMPLES}] {text[:150]:<10}")



RANDOM SAMPLE OF COMMENTS TO IDENTIFY ISSUES
[1/25] Can’t be bothered to pronounce the Asian names properly I guess.
[2/25] 14:30 in my opinion, it needed a big sumptuous cloak, maybe with a hood and gloves, to be more evocative of how luscious and involved klimt’s pieces a
[3/25] what is cara doing :/
[4/25] We need you at the Grammy asap😂😂
[5/25] @LadyAxe13 🤣
[6/25] Megyn is sooo jealous she wasn't invited
[7/25] My only look at the met gala, thanks Garrron! Getting Halloween in springtime vibes 🤡👻👽👀
[8/25] I agree with so many of your critiques. The fact Anna didn't even bother with the theme and wore a variation of a previous dress let's me know how fri
[9/25] I be sick of them shades. You can't wear shades with everything. I was just thinking this when I first seen this video. She look like one of the actor
[10/25] NINGNING  
[11/25] Vogue deletes comments and leaves hates comments on him.
[12/25] Exactly!  
[13/25] The idea is worth sharing This deserves recognition.
[14/25] Bla

In [86]:
def detect_language(text):
    """Detect language, returning ISO code."""
    try:
        if not text or not len(text.strip()):
            return 'en'
        return detect(text)
    except LangDetectException:
        return 'unknown'

# Collect comment rows from the rows list
language_results = [(comment, detect_language(comment.get('comment_text', ''))) for comment in comments_flattened]
language_counter = Counter(lang for _, lang in language_results)

In [87]:
TOP_LANGUAGE_COUNT = 10
TOTAL_NON_ENGLISH_EXAMPLES = 20

total_comments = len(language_results)

print(f"\nTOP {TOP_LANGUAGE_COUNT} LANGUAGE DETECTIONS")
print("=" * 80)
for i, (lang, count) in enumerate(language_counter.most_common(TOP_LANGUAGE_COUNT), start=1):
    pct = 100 * count / total_comments
    print(f"[{i}] {lang} {count:,} ({pct:.2f}%)")
    if i == 10:
        break


TOP 10 LANGUAGE DETECTIONS
[1] en 46,961 (74.25%)
[2] unknown 1,679 (2.65%)
[3] so 1,563 (2.47%)
[4] pt 1,142 (1.81%)
[5] de 1,058 (1.67%)
[6] tl 1,049 (1.66%)
[7] af 973 (1.54%)
[8] fr 854 (1.35%)
[9] et 792 (1.25%)
[10] id 762 (1.20%)


In [169]:
COMMENT_TRUNCATION_LENGTH = 200

# Extract full comment data for English comments
english_comments = [comment for comment, lang in language_results if lang == 'en']
# Overwrite comments flattened with English filtered list
comments_flattened = english_comments

# Extract truncated comment data for non-English comments for test display
non_english_comments = [comment['comment_text'][:COMMENT_TRUNCATION_LENGTH] for comment, lang in language_results if lang != 'en']

random.seed(RANDOM_SEED)
random_samples = random.sample(non_english_comments, min(TOTAL_NON_ENGLISH_EXAMPLES, len(non_english_comments)))

print("\nRANDOM NON-ENGLISH COMMENTS REMOVED:")
print("=" * 80)
for idx, text in enumerate(random_samples):
    print(f"[{idx+1}/{TOTAL_NON_ENGLISH_EXAMPLES}] {text}")



RANDOM NON-ENGLISH COMMENTS REMOVED:
[1/20] Aam ki tokri lg rhi hr😂
[2/20] Circlejerk
[3/20] Ningning guennn
[4/20] Lembra mais o pai dele do que Michael
[5/20] Absolute madness 😅😂
[6/20] Best review ever
[7/20] 13:42 hahaha this was a good one
[8/20] QUEEN BEYONCÉ 🤍🤍🤍🤍🤍🤍🤍AND BLUE 💙💙💙💙💙💙💙💙💙💙💙💙💙💙💙
SO BEAUTYFULLS
[9/20] Splendido ❤
[10/20] Here ❤
[11/20] La que es Bella🌟 es Bella ⭐nunca nesecita ninguna Luz para Brillar🌟
🌺💐🌺💐🌺💐🌺
[12/20] Bravissimo 👏👏👏
[13/20] Wake up world
[14/20] "It's giving La Larona" 😆😆😆
[15/20] I see BLACKPINK. And I'm happy Blinks ✨🩷
[16/20] @freshstrt3140 lmao you people are delusional af! 😂
[17/20] Fifth okk
[18/20] Léna ?
[19/20] Donnatella Versace
[20/20] Dünya yanıyor bunlar galada gülücük dağıtıyor


In [112]:
english_count = len(english_comments)
removed_count = total_comments - english_count
english_pct = 100 * english_count / total_comments
removed_pct = 100 * removed_count / total_comments

print(f"\nEnglish kept: {english_count} ({english_pct:.1f}%)")
print(f"Non-English removed: {removed_count} ({removed_pct:.1f}%)")


English kept: 46961 (74.2%)
Non-English removed: 16289 (25.8%)


In [113]:
# Regex patterns shared by the cleaning helpers
URL_PATTERN = re.compile(r'https?://\S+')
TIMESTAMP_PATTERN = re.compile(r'\b\d{1,2}:\d{2}(?::\d{2})?\b')
MENTION_PATTERN = re.compile(r'@[\w.-]+[\w]')
DIGIT_PATTERN = re.compile(r'\d+')
PUNCT_PATTERN = re.compile(r'[^\w\s]')


In [127]:
# Light cleaning for entity matching and network basic network assessment
def remove_html_entities(text):
    return html.unescape(text or "")

def remove_urls(text):
    return URL_PATTERN.sub("", text)

def remove_timestamps(text):
    return TIMESTAMP_PATTERN.sub("", text)

def remove_mentions(text):
    return MENTION_PATTERN.sub("", text)

def normalise_whitespace(text):
    return " ".join(text.split())

def clean_for_entity_matching(text):
    text = remove_html_entities(text)
    text = remove_urls(text)
    text = remove_timestamps(text)
    text = remove_mentions(text)
    text = text.lower().strip()
    return normalise_whitespace(text)

# Store minimally processed text
# Depending on sentiment approach, this might be enough processing
for comment in comments_flattened:
    comment["comment_text_entity"] = clean_for_entity_matching(comment.get("comment_text", ""))

print("Lightly cleaned text ready for entity matching")


Lightly cleaned text ready for entity matching


In [170]:
unique_videos = set(comment['video_id'] for comment in comments_flattened)
unique_channels = set(comment['channel_id'] for comment in comments_flattened)
unique_authors = set(comment['comment_author'] for comment in comments_flattened)

SHORT_COMMENT_LENGTH = 8
short_comments = [comment for comment in comments_flattened if len(comment['comment_text']) < SHORT_COMMENT_LENGTH]
comment_lengths = [len(comment['comment_text']) for comment in comments_flattened]

comments_with_urls = [c for c in comments_flattened if URL_PATTERN.search(c['comment_text'])]
comments_with_timestamps = [c for c in comments_flattened if TIMESTAMP_PATTERN.search(c['comment_text'])]
comments_with_mentions = [c for c in comments_flattened if MENTION_PATTERN.search(c['comment_text'])]
comments_with_digits = [c for c in comments_flattened if DIGIT_PATTERN.search(c['comment_text'])]
comments_with_punctuation = [c for c in comments_flattened if PUNCT_PATTERN.search(c['comment_text'])]

comments_with_urls_pct = 100 * len(comments_with_urls) / len(comments_flattened)
comments_with_timestamps_pct = 100 * len(comments_with_timestamps) / len(comments_flattened)
comments_with_mentions_pct = 100 * len(comments_with_mentions) / len(comments_flattened)
comments_with_digits_pct = 100 * len(comments_with_digits) / len(comments_flattened)
comments_with_punctuation_pct = 100 * len(comments_with_punctuation) / len(comments_flattened)

# For author, video, and channel distributions
video_counter = Counter(comment['video_id'] for comment in comments_flattened)
channel_counter = Counter(comment['channel_id'] for comment in comments_flattened)
author_counter = Counter(comment['comment_author'] for comment in comments_flattened)

# Print all details at bottom
print("BASIC DATA EXPLORATION")
print("=" * 80)
print(f"Total comments: {len(comments_flattened)}")
print(f"Unique videos: {len(unique_videos)}")
print(f"Unique channels: {len(unique_channels)}")
print(f"Unique authors: {len(unique_authors)}\n")

print(f"Short comments (<{SHORT_COMMENT_LENGTH} chars): {len(short_comments)}")
print(f"Comments w/ URLs: {len(comments_with_urls)} ({comments_with_urls_pct:.2f}%)")
print(f"Comments w/ timestamps: {len(comments_with_timestamps)} ({comments_with_timestamps_pct:.2f}%)")
print(f"Comments w/ mentions: {len(comments_with_mentions)} ({comments_with_mentions_pct:.2f}%)")
print(f"Comments w/ digits: {len(comments_with_digits)} ({comments_with_digits_pct:.2f}%)")
print(f"Comments w/ punctuation: {len(comments_with_punctuation)}\n")

print(f"Max comment length: {max(comment_lengths) if comment_lengths else 0}")
print(f"Average comment length: {sum(comment_lengths)/len(comment_lengths):.2f}" if comment_lengths else "Avg. comment length: 0")

BASIC DATA EXPLORATION
Total comments: 46961
Unique videos: 109
Unique channels: 73
Unique authors: 35150

Short comments (<8 chars): 231
Comments w/ URLs: 19 (0.04%)
Comments w/ timestamps: 1466 (3.12%)
Comments w/ mentions: 3770 (8.03%)
Comments w/ digits: 6921 (14.74%)
Comments w/ punctuation: 40189

Max comment length: 9817
Average comment length: 97.73


In [171]:
with open(MET_GALA_ENTITIES_PATH, "r", encoding="utf-8") as f:
    met_gala_entities = json.load(f)

total_entities = len(met_gala_entities['entities'])
celebs = [entity for entity in met_gala_entities['entities'].values() if entity['type'] == 'celebrity']
brands = [entity for entity in met_gala_entities['entities'].values() if entity['type'] == 'designer_brand']

print(f"Total entities: {total_entities}")
print(f"Total celebrities: {len(celebs)}")
print(f"Total brands: {len(brands)}")

Total entities: 483
Total celebrities: 361
Total brands: 122


In [ ]:
# Prepare entity alias patterns after light cleaning is available
def build_entity_patterns(entities):
    entity_patterns = []
    for entity in entities:
        patterns = []
        for alias in entity.get("aliases", []):
            
            alias = clean_for_entity_matching(alias)
            if not alias:
                continue
            pattern = rf"(?<![a-z0-9]){re.escape(alias)}(?![a-z0-9])"
            patterns.append(re.compile(pattern))
        entity_patterns.append({"name": entity["name"], "patterns": patterns})
    return entity_patterns

def find_entities(clean_text, entity_patterns):
    found = []
    for entity in entity_patterns:
        for pattern in entity["patterns"]:
            if pattern.search(clean_text):
                found.append(entity["name"])
                break
    return found

celeb_patterns = build_entity_patterns(celebs)
brand_patterns = build_entity_patterns(brands)


In [ ]:
# Match celebrity and brand aliases in each comment
matched_comments = []
celeb_counter = Counter()
brand_counter = Counter()

for comment in comments_flattened:
    text_for_matching = comment.get("comment_text_entity", "")

    # Find entities in comment
    found_celebs = find_entities(text_for_matching, celeb_patterns)
    found_brands = find_entities(text_for_matching, brand_patterns)

    # Scoring on each counter
    for celeb in found_celebs:
        celeb_counter[celeb] += 1
    for brand in found_brands:
        brand_counter[brand] += 1

    # Append to matched comments
    matched_comments.append({
        "comment_id": comment.get("comment_id"),
        "video_id": comment.get("video_id"),
        "video_title": comment.get("video_title"),
        "comment_text": comment.get("comment_text", ""),
        "comment_text_entity": text_for_matching,
        "celebs": found_celebs,
        "brands": found_brands,
    })


In [174]:
comments_with_celebs = [row for row in matched_comments if row["celebs"]]
comments_with_brands = [row for row in matched_comments if row["brands"]]
# Which comments have both brand and celeb mentions 
# Expect this to be smaller
comments_with_both = [row for row in matched_comments if row["celebs"] and row["brands"]]

comments_with_celebs_pct = 100 * len(comments_with_celebs) / len(comments_flattened)
comments_with_brands_pct = 100 * len(comments_with_brands) / len(comments_flattened)
comments_with_both_pct = 100 * len(comments_with_both) / len(comments_flattened)

print("ENTITY MATCH COVERAGE")
print("=" * 80)
print(f"Total comments: {len(comments_flattened)}")
print(f"Comments with celebrity mentions: {len(comments_with_celebs)} ({comments_with_celebs_pct:.2f}%)")
print(f"Comments with brand mentions: {len(comments_with_brands)} ({comments_with_brands_pct:.2f}%)")
print(f"Comments with both celebrity and brand mentions: {len(comments_with_both)} ({comments_with_both_pct:.2f}%)")


ENTITY MATCH COVERAGE
Total comments: 46961
Comments with celebrity mentions: 11160 (23.76%)
Comments with brand mentions: 1024 (2.18%)
Comments with both celebrity and brand mentions: 377 (0.80%)


In [175]:
# Entities that are being found often enough to work with
matched_celeb_count = len(celeb_counter)
matched_brand_count = len(brand_counter)
matched_celeb_count_pct = 100 * matched_celeb_count / len(celebs)
matched_brand_count_pct = 100 * matched_brand_count / len(brands)

print("ENTITY FREQUENCY CHECK")
print("=" * 80)
print(f"Matched celebrities: {matched_celeb_count}/{len(celebs)} ({matched_celeb_count_pct:.2f}%)")
print(f"Matched brands: {matched_brand_count}/{len(brands)} ({matched_brand_count_pct:.2f}%)")

print("\nTOP 20 CELEBRITIES")
print("=" * 80)
for celeb, count in celeb_counter.most_common(20):
    print(f"{celeb}: {count}")

print("\nTOP 20 BRANDS")
print("=" * 80)
for brand, count in brand_counter.most_common(20):
    print(f"{brand}: {count}")


ENTITY FREQUENCY CHECK
Matched celebrities: 214/361 (59.28%)
Matched brands: 66/122 (54.10%)

TOP 20 CELEBRITIES
Beyonce: 1544
Jisoo: 1306
LISA: 1089
Rose: 866
Rihanna: 775
Kim Kardashian: 595
Madonna: 464
JENNIE: 422
Emma Chamberlain: 380
Cardi B: 350
Heidi Klum: 348
Anne Hathaway: 326
Kylie Jenner: 266
Katy Perry: 242
Bad Bunny: 234
Blake Lively: 216
Sabrina Carpenter: 215
Karan Johar: 203
Tyla: 191
Sam Smith: 190

TOP 20 BRANDS
Saint Laurent: 258
Robert Wun: 162
Dior: 89
Mugler: 76
Chanel: 70
Hugo Boss: 52
Balenciaga: 43
Schiaparelli: 41
Prada: 32
Zara: 24
Skims: 21
Allen Jones: 20
Zac Posen: 17
Gap Studio: 16
Valentino: 15
Chloe: 13
Tom Ford: 10
Viktor & Rolf: 10
Alexander McQueen: 9
Christian Siriano: 9


In [176]:
# Count tuple pairs of (brand, celeb)
brand_celeb_counter = Counter()
for row in comments_with_both:
    for brand in row["brands"]:
        for celeb in row["celebs"]:
            brand_celeb_counter[(brand, celeb)] += 1

# Create edge rows
edge_rows = []
for (brand, celeb), count in brand_celeb_counter.items():
    edge_rows.append({
        "source": brand,
        "target": celeb,
        "source_type": "brand",
        "target_type": "celebrity",
        "weight": count,
    })

# Higher weight better, stronger indicator
edge_rows_sorted = sorted(edge_rows, key=lambda x: x["weight"], reverse=True)

In [177]:
TOTAL_BRAND_CELEBRITY_EXAMPLES = 40

print("BRAND-CELEBRITY EDGE CHECK")
print("=" * 80)
print(f"Unique brand-celebrity edges: {len(edge_rows_sorted)}")
print(f"Total brand-celebrity co-mentions: {sum(brand_celeb_counter.values())}")

BRAND-CELEBRITY EDGE CHECK
Unique brand-celebrity edges: 429
Total brand-celebrity co-mentions: 876


In [178]:

print("\nTOP 20 BRAND-CELEBRITY EDGES")
BRAND_CELEBRITY_PAD = 40
WEIGHT_PAD = 10
print(f"{'(BRAND, CELEBRITY)':<{BRAND_CELEBRITY_PAD}} {'WEIGHT':<{WEIGHT_PAD}}")
print("=" * (BRAND_CELEBRITY_PAD + 1 + WEIGHT_PAD))
for edge in edge_rows_sorted[:TOTAL_BRAND_CELEBRITY_EXAMPLES]:
    source_target = (edge['source'], edge['target'])
    print(f"{str(source_target):<{BRAND_CELEBRITY_PAD}} {edge['weight']:<{WEIGHT_PAD}}")



TOP 20 BRAND-CELEBRITY EDGES
(BRAND, CELEBRITY)                       WEIGHT    
('Saint Laurent', 'Rose')                58        
('Dior', 'Jisoo')                        30        
('Robert Wun', 'LISA')                   24        
('Mugler', 'Emma Chamberlain')           17        
('Allen Jones', 'Kim Kardashian')        14        
('Saint Laurent', 'Jisoo')               14        
('Dior', 'LISA')                         11        
('Saint Laurent', 'Connor Storrie')      11        
('Saint Laurent', 'LISA')                11        
('Chanel', 'JENNIE')                     10        
('Saint Laurent', 'JENNIE')              10        
('Robert Wun', 'Naomi Osaka')            9         
('Saint Laurent', 'Madonna')             7         
('Schiaparelli', 'Lauren Sanchez Bezos') 7         
('Dior', 'JENNIE')                       7         
('Dior', 'Rose')                         7         
('Balenciaga', 'Beyonce')                7         
('Chloe', 'Chloe Malle')          

In [179]:
# Basic inspection of nodes and edges
node_rows = []
for celeb, count in celeb_counter.items():
    node_rows.append({
        "node": celeb, 
        "type": "celebrity", 
        "mention_count": count
    })
for brand, count in brand_counter.items():
    node_rows.append({
        "node": brand, 
        "type": "brand", 
        "mention_count": count
    })
node_rows_sorted = sorted(node_rows, key=lambda x: x["mention_count"], reverse=True)

print("BASIC NETWORK ASSESSMENT")
print("=" * 80)
print(f"Nodes available: {len(node_rows_sorted)}")
print(f"Edges available: {len(edge_rows_sorted) if 'edge_rows_sorted' in locals() else 0}")
print(f"Comments supporting edges: {len(comments_with_both)}")
print(f"Edges with weight >= 2: {sum(1 for edge in edge_rows_sorted if edge['weight'] >= 2) if 'edge_rows_sorted' in locals() else 0}")


BASIC NETWORK ASSESSMENT
Nodes available: 280
Edges available: 429
Comments supporting edges: 377
Edges with weight >= 2: 131


In [180]:
TWEET_TOKENISER = nltk.tokenize.TweetTokenizer(
    reduce_len=True,
    strip_handles=True,
    preserve_case=False 
)

PUNCTUATION = list(string.punctuation)
TWEET_STEMMER = nltk.stem.PorterStemmer()
STOP_WORDS = set(stopwords.words('english')) | set(PUNCTUATION)


In [181]:
# Heavier cleaning for later lexical-based sentiment/topic preprocessing
def remove_digits(text):
    return DIGIT_PATTERN.sub(" ", text)

def remove_unicode(text):
    return text.encode("ascii", "ignore").decode()

def strip_punctuation(text):
    return PUNCT_PATTERN.sub(" ", text)

def tokenize(text):
    return TWEET_TOKENISER.tokenize(text)

def remove_stopwords(tokens):
    return [t for t in tokens if t not in STOP_WORDS]

def stem_tokens(tokens):
    return [TWEET_STEMMER.stem(t) for t in tokens]

def remove_emojis(text):
    return emoji.replace_emoji(text, replace="")

def clean_for_sentiment_tokens(text):
    text = clean_for_entity_matching(text)
    text = remove_unicode(text)
    text = remove_digits(text)
    text = strip_punctuation(text)
    text = normalise_whitespace(text)
    tokens = tokenize(text)
    tokens = remove_stopwords(tokens)
    return stem_tokens(tokens)

def purify_text(text, show_changes=False):
    """Return heavily cleaned tokens for later sentiment/topic work."""
    if not show_changes:
        return clean_for_sentiment_tokens(text)

    history = {}
    text = remove_html_entities(text)
    history["remove_html_entities"] = text
    text = remove_urls(text)
    history["remove_urls"] = text
    text = remove_timestamps(text)
    history["remove_timestamps"] = text
    text = remove_mentions(text)
    history["remove_mentions"] = text
    text = text.lower().strip()
    history["lowercase_and_strip"] = text
    text = normalise_whitespace(text)
    history["normalise_whitespace"] = text
    text = remove_unicode(text)
    history["remove_unicode"] = text
    text = remove_digits(text)
    history["remove_digits"] = text
    text = strip_punctuation(text)
    history["strip_punctuation"] = text
    text = normalise_whitespace(text)
    history["normalise_whitespace_after_punctuation"] = text
    tokens = tokenize(text)
    history["tokenize"] = tokens
    tokens = remove_stopwords(tokens)
    history["remove_stopwords"] = tokens
    tokens = stem_tokens(tokens)
    history["stem_tokens"] = tokens
    return history


In [182]:
# Demonstration of processing for random sample and report
RANDOM_TOTAL_EXAMPLES = 10
purify_random_samples = random.sample(english_comments, RANDOM_TOTAL_EXAMPLES)
for idx, text in enumerate(purify_random_samples):
    history = purify_text(text['comment_text'], True).items()
    print(f"\n[{idx+1}/{RANDOM_TOTAL_EXAMPLES}] {text['comment_text'][:100]}")
    for step, value in history:
        print(f"  [{step}] {value}")


[1/10] Wow, that is certainly extra. Some of that "art" is impossible to sit in.
  [remove_html_entities] Wow, that is certainly extra. Some of that "art" is impossible to sit in.
  [remove_urls] Wow, that is certainly extra. Some of that "art" is impossible to sit in.
  [remove_timestamps] Wow, that is certainly extra. Some of that "art" is impossible to sit in.
  [remove_mentions] Wow, that is certainly extra. Some of that "art" is impossible to sit in.
  [lowercase_and_strip] wow, that is certainly extra. some of that "art" is impossible to sit in.
  [normalise_whitespace] wow, that is certainly extra. some of that "art" is impossible to sit in.
  [remove_unicode] wow, that is certainly extra. some of that "art" is impossible to sit in.
  [remove_digits] wow, that is certainly extra. some of that "art" is impossible to sit in.
  [strip_punctuation] wow  that is certainly extra  some of that  art  is impossible to sit in 
  [normalise_whitespace_after_punctuation] wow that is certai